In [1]:
import requests
from bs4 import BeautifulSoup
import pandas as pd
import numpy as np
import time
import random
import re
from tqdm import tqdm
import os

In [2]:
# Config
HEADERS = {
    'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36',
    'Accept': 'text/html,application/xhtml+xml,application/xml;q=0.9,image/webp,*/*;q=0.8',
    'Accept-Language': 'vi-VN,vi;q=0.9,en-US;q=0.8,en;q=0.7',
}

BASE_URL = 'https://www.dienmayxanh.com'
MIN_DELAY = 0.5
MAX_DELAY = 1.5

## 1. Lấy danh sách Categories

In [8]:
def get_dmx_categories():
    categories = [
        {'name': "Món canh", 'url': 'https://www.dienmayxanh.com/vao-bep/mon-canh'},
        {'name': "Món bánh", 'url': 'https://www.dienmayxanh.com/vao-bep/mon-banh'},
        {'name': "Món chè", 'url': 'https://www.dienmayxanh.com/vao-bep/mon-che'},
        {'name': "Món nước", 'url': 'https://www.dienmayxanh.com/vao-bep/mon-nuoc'},
        {'name': "Món kem", 'url': 'https://www.dienmayxanh.com/vao-bep/mon-kem'},
        {'name': "Món nướng", 'url': 'https://www.dienmayxanh.com/vao-bep/mon-nuong'},
        {'name': "Món cháo", 'url': 'https://www.dienmayxanh.com/vao-bep/mon-chao'},
        {'name': "Món xào", 'url': 'https://www.dienmayxanh.com/vao-bep/mon-xao'},

        {'name': "Món kho", 'url': 'https://www.dienmayxanh.com/vao-bep/mon-kho'},
        {'name': "Món chiên", 'url': 'https://www.dienmayxanh.com/vao-bep/mon-chien'},        
        {'name': "Món hấp", 'url': 'https://www.dienmayxanh.com/vao-bep/mon-hap'},
        {'name': "Món lẩu", 'url': 'https://www.dienmayxanh.com/vao-bep/mon-lau'},
        {'name': "Món gỏi - salad", 'url': 'https://www.dienmayxanh.com/vao-bep/mon-goi-tron'},
        {'name': "Món từ gà", 'url': 'https://www.dienmayxanh.com/vao-bep/mon-tu-ga'},
        {'name': "Món từ bò", 'url': 'https://www.dienmayxanh.com/vao-bep/mon-tu-bo'},
        {'name': "Món chay", 'url': 'https://www.dienmayxanh.com/vao-bep/mon-chay'},
        {'name': "Ăn vặt", 'url': 'https://www.dienmayxanh.com/vao-bep/an-vat'},

        {'name': "Ngày lễ Tết", 'url': 'https://www.dienmayxanh.com/vao-bep/ngay-le-tet'},
        {'name': "Thức uống", 'url': 'https://www.dienmayxanh.com/vao-bep/thuc-uong'},
        {'name': "Sinh tố", 'url': 'https://www.dienmayxanh.com/vao-bep/sinh-to'},
        {'name': "Trà sữa", 'url': 'https://www.dienmayxanh.com/vao-bep/tra-sua'},
        {'name': "Nước ép", 'url': 'https://www.dienmayxanh.com/vao-bep/nuoc-ep'},
        {'name': "Món tráng miệng", 'url': 'https://www.dienmayxanh.com/vao-bep/mon-trang-mieng'},
        {'name': "Món khô - mắm", 'url': 'https://www.dienmayxanh.com/vao-bep/mon-kho-mam'},
        {'name': "Món cuốn - trộn", 'url': 'https://www.dienmayxanh.com/vao-bep/mon-cuon-tron'}
    ]
    return categories

categories = get_dmx_categories()
print(f"Tổng số category: {len(categories)}")

Tổng số category: 25


In [15]:
categories

[{'name': 'Món canh', 'url': 'https://www.dienmayxanh.com/vao-bep/mon-canh'},
 {'name': 'Món bánh', 'url': 'https://www.dienmayxanh.com/vao-bep/mon-banh'},
 {'name': 'Món chè', 'url': 'https://www.dienmayxanh.com/vao-bep/mon-che'},
 {'name': 'Món nước', 'url': 'https://www.dienmayxanh.com/vao-bep/mon-nuoc'},
 {'name': 'Món kem', 'url': 'https://www.dienmayxanh.com/vao-bep/mon-kem'},
 {'name': 'Món nướng', 'url': 'https://www.dienmayxanh.com/vao-bep/mon-nuong'},
 {'name': 'Món cháo', 'url': 'https://www.dienmayxanh.com/vao-bep/mon-chao'},
 {'name': 'Món xào', 'url': 'https://www.dienmayxanh.com/vao-bep/mon-xao'},
 {'name': 'Món kho', 'url': 'https://www.dienmayxanh.com/vao-bep/mon-kho'},
 {'name': 'Món chiên', 'url': 'https://www.dienmayxanh.com/vao-bep/mon-chien'},
 {'name': 'Món hấp', 'url': 'https://www.dienmayxanh.com/vao-bep/mon-hap'},
 {'name': 'Món lẩu', 'url': 'https://www.dienmayxanh.com/vao-bep/mon-lau'},
 {'name': 'Món gỏi - salad',
  'url': 'https://www.dienmayxanh.com/vao-b

## 2. Lấy danh sách URL công thức

Do trang điện máy xanh có định dạng cần bấm nút "Xem thêm" thay vì duyệt qua các page nên ta cần dùng selenium để giả lập click nút xem thêm này

In [14]:
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.common.exceptions import TimeoutException, ElementClickInterceptedException
import time

def get_food_urls_from_category_dmx(category_url):
    driver = webdriver.Chrome()
    driver.get(category_url)
    wait = WebDriverWait(driver, 10)

    food_urls = set()   # tránh trùng lặp

    prev_count = 0

    while True:
        # 1. Try to click "Xem thêm"
        try:
            see_more_btn = wait.until(
                EC.element_to_be_clickable((By.CSS_SELECTOR, "a.seemore-cook"))
            )
            driver.execute_script("arguments[0].scrollIntoView(true);", see_more_btn)
            time.sleep(0.3)
            see_more_btn.click()
        except TimeoutException:
            # Không còn nút => load hết rồi
            break
        except ElementClickInterceptedException:
            # Thử lại nhẹ
            time.sleep(1)
            continue

        # 2. Wait until new li are loaded
        time.sleep(1)  # nhẹ để JS load
        all_items = driver.find_elements(By.CSS_SELECTOR, "ul li a")
        cur_count = len(all_items)

        if cur_count == prev_count:
            # Không tăng nữa => hết dữ liệu
            break

        prev_count = cur_count

    # 3. Extract URL từ tất cả li
    final_items = driver.find_elements(By.CSS_SELECTOR, "ul li a")
    for item in final_items:
        href = item.get_attribute("href")
        if href and "/vao-bep/" in href:   # filter chuẩn
            food_urls.add(href)

    driver.quit()
    return list(food_urls)

In [19]:
# Test trên 1 category
print(f"Testing: {categories[0]['name']}")

test_urls = get_food_urls_from_category_dmx(categories[0]['url'])
print(f"Found {len(test_urls)} recipes")

Testing: Món canh
Found 960 recipes


In [20]:
test_urls[:3]

['https://www.dienmayxanh.com/vao-bep/2-cach-nau-canh-rau-ngot-chay-dau-hu-thom-ngon-thanh-mat-don-11520',
 'https://www.dienmayxanh.com/vao-bep/cach-nau-ga-ac-tan-ham-ngai-cuu-cuc-ngon-bo-mau-huyet-01384',
 'https://www.dienmayxanh.com/vao-bep/3-cach-nau-canh-dua-leo-xuong-heo-nhoi-thit-va-tom-thanh-mat-05075']

In [21]:
# Cào tất cả categories
all_recipe_urls = []
seen_urls = set()

for category in tqdm(categories, desc="Crawling categories"):
    cat_name = category["name"]
    cat_url  = category["url"]

    print(f"Crawling category: {cat_name}")

    try:
        food_urls = get_food_urls_from_category_dmx(cat_url)
    except Exception as e:
        print(f"Lỗi khi crawl category {cat_name}: {e}")
        continue

    for url in food_urls:
        if url not in seen_urls:
            seen_urls.add(url)
            all_recipe_urls.append((cat_name, url))

    print(f"Total collected: {len(all_recipe_urls)}")
    time.sleep(random.uniform(1, 2))

print(f"Total unique recipes: {len(all_recipe_urls)}")


Crawling categories:   0%|          | 0/25 [00:00<?, ?it/s]

Crawling category: Món canh
Total collected: 960


Crawling categories:   4%|▍         | 1/25 [01:30<36:10, 90.45s/it]

Crawling category: Món bánh
Total collected: 3518


Crawling categories:   8%|▊         | 2/25 [08:08<1:44:07, 271.63s/it]

...

Crawling category: Món khô - mắm
Total collected: 13394


Crawling categories:  96%|█████████▌| 24/25 [33:46<00:55, 55.51s/it]

Crawling category: Món cuốn - trộn
Total collected: 13515


Crawling categories: 100%|██████████| 25/25 [34:34<00:00, 82.99s/it]

Total unique recipes: 13515


In [23]:
all_food_urls = [(cat, url) for cat, url in all_recipe_urls]


In [24]:
all_food_urls

[('Món canh',
  'https://www.dienmayxanh.com/vao-bep/2-cach-nau-canh-rau-ngot-chay-dau-hu-thom-ngon-thanh-mat-don-11520'),
 ('Món canh',
  'https://www.dienmayxanh.com/vao-bep/cach-nau-ga-ac-tan-ham-ngai-cuu-cuc-ngon-bo-mau-huyet-01384'),
 ('Món canh',
  'https://www.dienmayxanh.com/vao-bep/3-cach-nau-canh-dua-leo-xuong-heo-nhoi-thit-va-tom-thanh-mat-05075'),
 ('Món canh',
  'https://www.dienmayxanh.com/vao-bep/cach-nau-sup-nam-can-tay-de-lam-de-an-thom-nuc-mui-07891'),
 ('Món canh',
  'https://www.dienmayxanh.com/vao-bep/cach-nau-sup-cua-chay-thom-ngon-don-gian-02693'),
 ('Món canh',
  'https://www.dienmayxanh.com/vao-bep/cac-mon-canh-mua-dong-mien-bac-de-nau-giu-am-co-the-23616'),
 ('Món canh',
  'https://www.dienmayxanh.com/vao-bep/cach-nau-gio-heo-ham-cu-cai-muoi-thom-ngon-bo-duong-ca-nha-07674'),
 ('Món canh',
  'https://www.dienmayxanh.com/vao-bep/cach-nau-canh-ca-minh-thai-han-quoc-bo-duong-la-mieng-cuc-09540'),
 ('Món canh',
  'https://www.dienmayxanh.com/vao-bep/cach-lam-ca-me

In [25]:
all_foods_df = pd.DataFrame(all_food_urls, columns=['category', 'url'])
all_foods_df

,category,url
0,Món canh,https://www.dienmayxanh.com/vao-bep/2-cach-nau...
1,Món canh,https://www.dienmayxanh.com/vao-bep/cach-nau-g...
2,Món canh,https://www.dienmayxanh.com/vao-bep/3-cach-nau...
3,Món canh,https://www.dienmayxanh.com/vao-bep/cach-nau-s...
4,Món canh,https://www.dienmayxanh.com/vao-bep/cach-nau-s...
...,...,...
13510,Món cuốn - trộn,https://www.dienmayxanh.com/vao-bep/2-cach-lam...
13511,Món cuốn - trộn,https://www.dienmayxanh.com/vao-bep/cach-lam-c...
13512,Món cuốn - trộn,https://www.dienmayxanh.com/vao-bep/cach-lam-m...
13513,Món cuốn - trộn,https://www.dienmayxanh.com/vao-bep/cach-lam-c...


In [26]:
all_foods_df.to_csv('dienmayxanh_foods_urls.csv', index=False)
print("DataFrame successfully saved to output.csv")

DataFrame successfully saved to output.csv


## 3. Hàm crawl chi tiết công thức

In [16]:
#Load data từ csv
all_foods_df = pd.read_csv('dienmayxanh_foods_urls.csv')
all_foods_df.head()

,category,url
0,Món canh,https://www.dienmayxanh.com/vao-bep/2-cach-nau...
1,Món canh,https://www.dienmayxanh.com/vao-bep/cach-nau-g...
2,Món canh,https://www.dienmayxanh.com/vao-bep/3-cach-nau...
3,Món canh,https://www.dienmayxanh.com/vao-bep/cach-nau-s...
4,Món canh,https://www.dienmayxanh.com/vao-bep/cach-nau-s...


In [17]:
## Hàm để chỉ lấy text trong những đoạn có vừa text vừa link
def safe_text(node):
    if not node:
        return None

    p = node.find("p")
    if p:
        return p.get_text(" ", strip=True)

    return node.get_text(" ", strip=True)


In [18]:
import re

def normalize_time(text):
    """
    Chuẩn hoá thời gian tiếng Việt sang phút.
    """
    if not text:
        return None
    
    text = text.lower().strip()

    hours = 0
    minutes = 0

    # Bắt "x giờ" hoặc "xh"
    h1 = re.search(r'(\d+)\s*giờ', text)
    h2 = re.search(r'(\d+)\s*h(?!\w)', text)  # '1h', '2h'
    if h1:
        hours = int(h1.group(1))
    elif h2:
        hours = int(h2.group(1))

    # Bắt "y phút" hoặc "yph"
    m1 = re.search(r'(\d+)\s*phút', text)
    m2 = re.search(r'(\d+)\s*ph', text)
    if m1:
        minutes = int(m1.group(1))
    elif m2:
        minutes = int(m2.group(1))

    total = hours * 60 + minutes
    return str(total) + " phút" if total > 0 else None


In [19]:
def get_food_detail_dmx(url, category):
    result = {
        "link": url,
        "type_of_food": category,
        "title": None,
        "description": None,
        "author_name": None,
        "cook_time": None,
        "num_of_people": None,
        "calories": None,
        "num_of_ingredients": None,
        "ingredients": [],
        "step": [],
        "note": [],
        "post_date": None,
    }

    try:
        response = requests.get(url, headers=HEADERS, timeout=15)
        response.encoding = 'utf-8'
        soup = BeautifulSoup(response.text, 'html.parser')
        
        # Tìm vùng nội dung chính
        detail_content = soup.find('div', class_='detail-content')
        if not detail_content:
            detail_content = soup  # Fallback to toàn bộ page
        
        # Title - h1 đầu tiên trong detail-content
        try:
            title_tag = detail_content.find('h1')
            if title_tag:
                result['title'] = safe_text(title_tag)
        except:
            pass
        
        # Description - div.leadpost p
        try:
            leadpost = detail_content.find('div', class_='leadpost')
            if leadpost:
                result['description'] = safe_text(leadpost)
            else:
                # Fallback to meta description
                meta_desc = soup.find('meta', attrs={'name': 'description'})
                if meta_desc:
                    result['description'] = meta_desc.get('content', '')
        except:
            pass
        
        # Author - thử tìm nhiều vị trí
        try:
            # Từ script JSON-LD
            script_tag = soup.find('script', type='application/ld+json')
            if script_tag:
                import json
                try:
                    data = json.loads(script_tag.string)
                    if isinstance(data, dict) and 'author' in data:
                        author_data = data.get('author', {})
                        if isinstance(author_data, dict):
                            result['author_name'] = author_data.get('name')
                        elif isinstance(author_data, str):
                            result['author_name'] = author_data
                except:
                    pass
            
            if not result['author_name']:
                author_tag = soup.find('span', class_='author') or soup.find('a', class_='author')
                result['author_name'] = safe_text(author_tag)
        except:
            pass
        
        # Thông tin thời gian chế biến
        try:
            ready_items = soup.select('ul.ready li')
            for item in ready_items:
                h2_tag = item.find('h2')
                span_tag = item.find('span')
                if h2_tag and span_tag:
                    label = safe_text(h2_tag).lower()
                    value = safe_text(span_tag)

                    # Chỉ lấy "chế biến"
                    if 'chế biến' in label:
                        result['cook_time'] = normalize_time(value)
        except:
            pass

        # Số người - từ div.staple h2 small
        try:
            staple_div = soup.find('div', class_='staple')
            if staple_div:
                h2_tag = staple_div.find('h2')
                if h2_tag:
                    small_tag = h2_tag.find('small')
                    if small_tag:
                        raw_people = safe_text(small_tag).strip()

                        # Bỏ chữ "Cho"
                        raw_people = raw_people.lower()
                        if raw_people.startswith("cho"):
                            raw_people = raw_people[3:].strip()

                        result['num_of_people'] = raw_people
        except:
            pass
        
        # Ingredients - div.staple span
        try:
            ingredients = []
            staple_div = soup.find('div', class_='staple')
            if staple_div:
                ing_spans = staple_div.find_all('span', recursive=False)
                for span in ing_spans:
                    # Lấy tên nguyên liệu
                    name_parts = []
                    for content in span.contents:
                        if isinstance(content, str):
                            text = content.strip()
                            if text:
                                name_parts.append(text)
                    
                    name = ' '.join(name_parts).strip()
                    
                    # Lấy số lượng từ small
                    small_tag = span.find('small')
                    quantity = safe_text(small_tag) if small_tag else ''
                    
                    # Lấy ghi chú từ em
                    em_tag = span.find('em')
                    note = safe_text(em_tag) if em_tag else ''
                    
                    # Ghép lại
                    if name:
                        ingredient_text = name
                        if quantity:
                            ingredient_text = f"{quantity}" + f" {ingredient_text}"
                        if note:
                             ingredient_text += f" {note}"
                        ingredients.append(ingredient_text.strip())
            
            result['ingredients'] = ingredients
            result['num_of_ingredients'] = len(ingredients)
        except:
            pass
        
        # Steps - div.method ul li
        try:
            steps = []
            method_div = soup.find('div', class_='method')
            if method_div:
                step_items = method_div.find_all('li')
                for item in step_items:
                    # Lấy số bước từ label
                    label_tag = item.find('label')
                    step_num = safe_text(label_tag) if label_tag else ''
                    
                    # Lấy tiêu đề bước từ h3
                    h3_tag = item.find('h3')
                    step_title = safe_text(h3_tag) if h3_tag else ''
                    
                    # Lấy nội dung từ div.text-method
                    text_div = item.find('div', class_='text-method')
                    if text_div:
                        # Lấy tất cả p tags
                        p_tags = text_div.find_all('p')
                        contents = []
                        for p in p_tags:
                            text = safe_text(p)
                            if text:
                                contents.append(text)
                        step_content = ' '.join(contents)
                    else:
                        step_content = safe_text(item)
                    
                    # Ghép thành 1 bước
                    if step_content:
                        if step_num and step_title:
                            step_text = f"Bước {step_num}: {step_title}: {step_content}"
                        elif step_num:
                            step_text = f"Bước {step_num}: {step_content}"
                        else:
                            step_text = step_content
                        steps.append(step_text)
            
            result['step'] = steps
        except:
            pass
        
        # Notes - div.tipsrecipe
        try:
            notes = []
            tips_divs = soup.select('div.tipsrecipe, div.infobox, div.tips')
            for tips in tips_divs:
                text = safe_text(tips)
                if text:
                    # Loại bỏ "Mách nhỏ:" prefix
                    text = re.sub(r'^Mách nhỏ:\s*', '', text)
                    notes.append(text)
            result['note'] = notes
        except:
            pass
        
        # Post date - thử tìm từ nhiều nguồn
        try:
            # Từ meta hoặc span.date
            date_tag = soup.find('span', class_='date') or soup.find('time')
            if date_tag:
                result['post_date'] = safe_text(date_tag)
        except:
            pass
        
        return result
        
    except Exception as e:
        print(f"  [ERROR] {url}: {e}")
        return result

In [20]:
## Test trên 1 URL
test_url = all_foods_df.iloc[1]['url']
test_category = all_foods_df.iloc[1]['category']
test_food = get_food_detail_dmx(test_url, test_category)
test_food

{'link': 'https://www.dienmayxanh.com/vao-bep/cach-nau-ga-ac-tan-ham-ngai-cuu-cuc-ngon-bo-mau-huyet-01384',
 'type_of_food': 'Món canh',
 'title': 'Cách hầm gà lá ngải bằng nồi cơm điện đậm đà bổ dưỡng cho sức khỏe cả nhà',
 'description': 'Nếu bạn cần một gợi ý cho một món từ gà thơm ngon bổ dưỡng thì món gà hầm lá ngải sẽ là một món tuyệt vời không thể bỏ qua. Cùng Điện máy XANH vào bếp và thực hiện món canh này và mời người cùng thưởng thức ngay nào.',
 'author_name': None,
 'cook_time': '90 phút',
 'num_of_people': '3 người',
 'calories': None,
 'num_of_ingredients': 7,
 'ingredients': ['1 con Gà (khoảng 1.3 kg)',
  '1 củ Gừng',
  '50 gr Táo đỏ',
  '20 gr Kỷ tử',
  '1 ít Ngải cứu khô',
  '100 gr Ngải cứu tươi',
  '1 ít Gia vị thông dụng (Muối/ đường)'],
 'step': ['Bước 1: Sơ chế nguyên liệu: Gà bạn làm sạch rồi chà xát gà với 1 muỗng canh muối để khử bớt mùi hôi, sau đó bạn rửa sạch một lần nữa. Tiếp theo bạn chặt gà thành những miếng vừa ăn. Gừng bạn gọt vỏ, rửa sạch và cắt lát. T

# 4. Áp dụng hàm crawl trên toàn bộ urls

In [21]:
import re

# Hàm lọc URL món ăn hợp lệ (tránh url của trang nhiều món)
def check_valid_format(url):
    slug = url.rstrip('/').split('/')[-1].lower()

    # Trang tổng hợp nhiều món – bắt đầu bằng số lượng món
    if re.match(r'^\d+-cach', slug):
        return False

    # Tong hop
    if slug.startswith("tong-hop-"):
        return False

    # Các dạng tổng hợp khác
    if "cac-mon" in slug or "nhung-mon" in slug or "mon-ngon" in slug:
        return False

    # Mặc định: hợp lệ
    return True


In [22]:
test_df = all_foods_df.tail(100)

for idx, row in test_df.iterrows():
    url = row['url']
    if not(check_valid_format(url)):
        print(f"{url} => Invalid")

https://www.dienmayxanh.com/vao-bep/3-cach-lam-sushi-cuon-bo-la-mieng-hap-dan-cuc-don-gian-10915 => Invalid
https://www.dienmayxanh.com/vao-bep/tong-hop-cac-nguyen-lieu-lam-pho-cuon-day-du-chuan-vi-ha-noi-23528 => Invalid
https://www.dienmayxanh.com/vao-bep/3-cach-lam-banh-da-tron-thom-ngon-la-mieng-doi-vi-cho-bua-an-09662 => Invalid
https://www.dienmayxanh.com/vao-bep/2-cach-lam-vo-banh-trang-cuon-cha-gio-don-gian-an-toan-04602 => Invalid
https://www.dienmayxanh.com/vao-bep/2-cach-pha-nuoc-cham-pho-cuon-dam-da-ngon-de-lam-chuan-vi-14117 => Invalid
https://www.dienmayxanh.com/vao-bep/3-cach-lam-mit-tron-da-nang-sieu-ngon-chuan-vi-de-lam-tai-nha-14511 => Invalid
https://www.dienmayxanh.com/vao-bep/2-cach-lam-goi-cuon-ca-ngu-kieu-thai-va-hat-quinoa-moi-la-don-05165 => Invalid
https://www.dienmayxanh.com/vao-bep/2-cach-lam-goi-cuon-thit-ga-thanh-mat-thom-ngon-don-gian-cho-05208 => Invalid
https://www.dienmayxanh.com/vao-bep/tong-hop-8-cach-lam-goi-cuon-thom-ngon-dinh-duong-cho-ca-nha-1301

In [ ]:
total = len(all_foods_df)
all_foods = []
failed_urls = []
CHECKPOINT = 100

for i, row in all_foods_df.iterrows():
    category = row['category']
    url = row['url']

    # Print header line
    print(f"[{i+1}/{total}] {url}")

    # 1) Skip sớm – không tốn request
    if not check_valid_format(url):
        print("   → INVALID SLUG → SKIPPED")
        failed_urls.append((category, url, "Invalid slug"))
        continue

    try:
        # 2) Crawl detail
        detail = get_food_detail_dmx(url, category)

        # 3) Validate kết quả
        if detail['step'] and len(detail['ingredients']) > 0:
            print("   → OK")
            all_foods.append(detail)
        else:
            print("   → MISSING DATA")
            failed_urls.append((category, url, "Missing data"))

        # 4) Checkpoint
        if (i + 1) % CHECKPOINT == 0:
            pd.DataFrame(all_foods).to_csv('dienmayxanh_foods_checkpoint.csv', index=False)
            print(f"[CHECKPOINT] Checkpoint saved ({len(all_foods)} foods)")

        # 5) Delay
        time.sleep(random.uniform(MIN_DELAY, MAX_DELAY))

    except Exception as e:
        print(f"   → ERROR: {e}")
        failed_urls.append((category, url, str(e)))

print(f"\nSuccess: {len(all_foods)}")
print(f"Failed: {len(failed_urls)}")


[1/13515] https://www.dienmayxanh.com/vao-bep/2-cach-nau-canh-rau-ngot-chay-dau-hu-thom-ngon-thanh-mat-don-11520
   → INVALID SLUG → SKIPPED
[2/13515] https://www.dienmayxanh.com/vao-bep/cach-nau-ga-ac-tan-ham-ngai-cuu-cuc-ngon-bo-mau-huyet-01384
   → OK
[3/13515] https://www.dienmayxanh.com/vao-bep/3-cach-nau-canh-dua-leo-xuong-heo-nhoi-thit-va-tom-thanh-mat-05075
   → INVALID SLUG → SKIPPED
[4/13515] https://www.dienmayxanh.com/vao-bep/cach-nau-sup-nam-can-tay-de-lam-de-an-thom-nuc-mui-07891
   → OK
[5/13515] https://www.dienmayxanh.com/vao-bep/cach-nau-sup-cua-chay-thom-ngon-don-gian-02693
   → OK
[6/13515] https://www.dienmayxanh.com/vao-bep/cac-mon-canh-mua-dong-mien-bac-de-nau-giu-am-co-the-23616
   → INVALID SLUG → SKIPPED
[7/13515] https://www.dienmayxanh.com/vao-bep/cach-nau-gio-heo-ham-cu-cai-muoi-thom-ngon-bo-duong-ca-nha-07674
   → OK
[8/13515] https://www.dienmayxanh.com/vao-bep/cach-nau-canh-ca-minh-thai-han-quoc-bo-duong-la-mieng-cuc-09540
   → OK
[9/13515] https://www.d

## Phần này đã được chạy ở trên KAGGLE. Link: https://www.kaggle.com/code/minhtien3103/crawl-dienmayxanh-food

# Chuyển thành dataframe và lưu xuống

In [1]:
dmx_df = pd.DataFrame(all_foods)

NameError: name 'pd' is not defined

In [ ]:
dmx_df.head()

In [ ]:
# Lưu kết quả
dmx_df.to_csv('dienmayxanh_foods_detail.csv', index=False)
print(f"Saved {len(dmx_df)} foods")

if failed_urls:
    pd.DataFrame(failed_urls, columns=['category', 'url', 'error']).to_csv('dienmayxanh_failed_urls.csv', index=False)

Saved 252 recipes
